In [1]:
import os
from openai import AzureOpenAI
from tqdm import tqdm
import time
import re
from sklearn.metrics import *
from krippendorff import alpha as k_alpha

# Define Endpoint and Query Function

In [2]:
api_key = os.getenv("AZURE_OPENAI_API_KEY_MODEL")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_MODEL")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_MODEL")
api_version = os.getenv("AZURE_OPENAI_API_VERSION_MODEL")

In [3]:
# client = AzureOpenAI(
#     azure_endpoint=endpoint,
#     api_key=api_key,
#     api_version=api_version,
# )

In [4]:
# response = client.chat.completions.create(
#     model=deployment_name,
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": "What is the capital of France?"}
#     ]
# )

In [5]:
# print(response.choices[0].message.content)

In [6]:
import os
from openai import AzureOpenAI

endpoint = ""
model_name = "gpt-5.2-chat"
deployment_name = "gpt-5.2-chat"

api_key = ""
api_version = ""

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=api_key,
)

response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "I am going to Paris, what should I see?",
        }
    ],
    max_completion_tokens =16384,
    model=deployment_name,
    reasoning_effort="none",
)

print(response.choices[0].message.content)

In [7]:
def query_llm(sys_msg, query_msg, dt=2e-1):
    time.sleep(dt)
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": query_msg}
            ],
            reasoning_effort="none",
        )
        return response.choices[0].message.content
    except Exception as e:
        print(e)
        return -1

# Load LLM Response Dataset

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [9]:
file_path = "sps_dataset_train_split.csv"

In [10]:
data = pd.read_csv(file_path)
data

# Query for LLM

In [11]:
all_criteria = ['Completeness', 'Clarity', 'Interpretability', 'Conciseness', 'Accuracy', 'Relevance']

criteria_prompts = {}
criteria_prompts["Completeness"] = """
Completeness is a criteria that judges whether the response is complete and comprehensive. 
That means a good response should have included all of the relevant information that answers the question.
Conversely, a poor response would miss out important details relevant to the question.
This is the exact scoring rubrics:
-2: "Important information is missing, causing major misunderstandings.",
-1: "Several details are missing, making the response only partially usable.",
0: "Mostly complete but lacking a few supporting details.",
1: "Complete with all necessary information and minimal omissions.",
2: "Fully comprehensive with all required details and no omissions."
"""
criteria_prompts["Clarity"] = """
Clarity is a criteria that judges whether the response is clear and easy to understand. 
That means a good response should be easily understandable and well phrased.
Conversely, a bad response would be poorly phrased with lack of focus and hence not easily understandable.
This is the exact scoring rubrics:
-2: "Very unclear and confusing, making it hard to understand.",
-1: "Partially unclear with awkward wording or ambiguous sentences.",
0: "Somewhat clear but with minor ambiguity or weak phrasing.",
1: "Clear, easy to follow, and well-phrased.",
2: "Extremely clear, well-articulated, and highly readable."
"""
criteria_prompts["Interpretability"] = """
Interpretability is a criteria that judges whether the response is interpretable and have a clear logic.
That means a good response should be self-consistent while having a logic that is easy to understand.
Conversely, a bad response would be very convoluted and difficult to understand, even if the logic is correct.
This is the exact scoring rubrics:
-2: "Difficult to understand with tangled reasoning or unclear logic.",
-1: "Partially understandable but with unclear logic or weak organization.",
0: "Generally understandable but occasionally confusing or inconsistent.",
1: "Easy to understand with clear logic and strong organization.",
2: "Extremely easy to understand, logically strong, and excellently organized."
"""
criteria_prompts["Conciseness"] = """
Conciseness is a criteria that judges whether the response is concise in writing.
That means a good response should be not contain any redundant phrasing or language.
Conversely, a bad resonse would be contain irrelevant words that inflates the length of the response. 
A bad response can usually be shortened by re-writing the response to have more concise English.
This is the exact scoring rubrics:
-2: "Very wordy, redundant, or filled with unnecessary details.",
-1: "Somewhat verbose with noticeable redundancy.",
0: "Some unnecessary wording but overall acceptable length.",
1: "Concise with minimal redundancy and clear expression.",
2: "Highly concise, focused, and free of all unnecessary words."
"""
criteria_prompts["Accuracy"] = """
Accuracy is a criteria that judges whether the response is accurate in content and does not contain false or misleading information.
That means a good response should be fully accurate with all of the information provided found in the given context.
A good response should not contain information not given in the context, even if they are common knowledge.
A good response should also avoid misleading statements, or statements that imply things that are not given in the context.
Conversely, a bad response would include incorrect or misleading statements, or statements that are based on pre-existing knowledge but not supported by the context.
This is the exact scoring rubrics:
-2: "Contains factually incorrect or fabricated information.",
-1: "Contains several factual inaccuracies or unclear claims.",
0: "Mostly accurate but with minor errors or ambiguous statements.",
1: "Accurate and reliable with no significant factual issues.",
2: "Fully precise, factually correct, and verifiable throughout."
"""
criteria_prompts["Relevance"] = """
Relevance is a criteria that judges whether the response is relevant to the question being asked.
That means a good response should be avoid unnecessary details that the question didn't ask about, even if they might be tangentially relevant.
A good response should not contain statements that are irrelevant to the question being asked, even if they are true and provided in the context.
Conversely, a bad response would include irrelevant statements, even if they are true based on the context.
This is the exact scoring rubrics:
-2: "Content is mostly irrelevant or off-topic.",
-1: "Content is partially irrelevant or only loosely connected to the topic.",
0: "Content is somewhat relevant but contains unnecessary or unfocused parts.",
1: "Content is relevant and contributes meaningfully to the topic.",
2: "Content is highly relevant, targeted, and fully aligned with the topic."
"""

In [12]:
def get_llm_input(text_chunk, question, answer):
    """
    Returns the system and query text to score the response
    """
    
    # Filter out non-ASCII characters
    text_chunk = re.sub(r'[^\x00-\x7f]',r'', text_chunk)
    question = re.sub(r'[^\x00-\x7f]',r'', question)
    # answer = re.sub(r'[^\x00-\x7f]',r'', answer)

    system_prompt = \
    f"""You are tasked with scoring a response to a question based on a context. You will score it based on 6 criterias.
    Your will be given a context, a question, and an answer. All three of them will be in text format.

    These are the 6 criterias: 
    {criteria_prompts["Completeness"]}
    {criteria_prompts["Clarity"]}
    {criteria_prompts["Interpretability"]}
    {criteria_prompts["Conciseness"]}
    {criteria_prompts["Accuracy"]}
    {criteria_prompts["Relevance"]}
    
    Finally, output only the score.

    For example, below is a possible set of input:
    This is the context given: Company A was created in 1992.
    This is the question asked: Which year was company A created?
    This is the given answer: Company A was created in year 1992 by Mr X, driven by the mission to achieve Y.
    
    This is a possible output:
    -1 2 0 1 1 -2
    """

    query_prompt = \
    f"""This is the context given: {text_chunk}
    This is the question asked: {question}
    This is the given answer: {answer}
    Score the answer based on the 6 given criterias. Output only the scores, separated by whitespace. Output nothing else. THe output should be 6 numbers.
    """
    return system_prompt, query_prompt

In [13]:
sys_msg, q_msg = get_llm_input(data['text_chunk'][0], data['question'][0], data['answer'][0])

In [14]:
sys_msg

In [15]:
response = query_llm(sys_msg=sys_msg, query_msg=q_msg)
response

# DSPY Attempt

In [16]:
import dspy
from dspy.teleprompt import *
import random
from sklearn.model_selection import train_test_split

In [17]:
lm = dspy.LM(f"{deployment_name}", api_key=api_key, base_url=endpoint, api_version=api_version, temperature=1, max_tokens=2048)
dspy.configure(lm=lm)

In [18]:
lm("Say this is a test!")  # => ['This is a test!']

In [19]:
program = dspy.Predict("instruction: str, prompt: str -> scores: str")
program

In [20]:
i = 14
sys_msg, q_msg = get_llm_input(data['text_chunk'][i], data['question'][i], data['answer'][i])
program(instruction=sys_msg, prompt=q_msg)

In [21]:
wanted_cols = ["generated_answer_Conciseness_score", "generated_answer_Interpretability_score", "generated_answer_Completeness_score", "generated_answer_Clarity_score", "generated_answer_Accuracy_score", "generated_answer_Relevance_score"]

In [22]:
data[wanted_cols].iloc[i].to_numpy().astype(int)

In [23]:
def get_index_from_answer(answer):
    return (data['answer'] == answer).idxmax()

In [24]:
def compute_lm_accuracy(example, prediction, trace=None):
    pred_scores = prediction.scores
    pred_scores = pred_scores.split(" ")
    if len(pred_scores) != 6:
        return -1
    try:
        for i in range(6):
            pred_scores[i] = int(pred_scores[i])
    except Exception as e:
        return -1
    pred_scores = np.array(pred_scores, dtype=int)
    
    # Extract the index based on the prompt
    prompt = example.prompt
    prompt = prompt.split("This is the given answer: ")[1]
    prompt = prompt.split("\n Score the answer based on the 6 given criterias. Output only the scores, separated by whitespace. Output nothing else. THe output should be 6 numbers.")[0]
    prompt = prompt.rstrip(" ").rstrip("\n")
    idx = get_index_from_answer(prompt)
    target = data[wanted_cols].iloc[idx].to_numpy().astype(int)

    return np.mean(target == pred_scores)

In [25]:
dataset = []
for i in range(data.shape[0]):
    sys_msg, q_msg = get_llm_input(data['text_chunk'][i], data['question'][i], data['answer'][i])
    dataset.append(dspy.Example(instruction=sys_msg, prompt=q_msg).with_inputs("instruction", "prompt"))

trainset, valset = train_test_split(dataset, test_size=0.2, random_state=3407)
len(trainset), len(valset)

In [26]:
opt = MIPROv2(
    metric=compute_lm_accuracy,
    # max_steps=16,
    # max_demos=8,
    auto="heavy",
    max_bootstrapped_demos=16,
    max_labeled_demos=8,
)

# compile/optimize the program (this only searches prompt/instruction space)
optimized_program = opt.compile(
    program, 
    trainset=trainset,
    valset=valset
)

optimized_program.save(f"./sps_dspy_program_gpt-5point2-chat/", save_program=True)

In [39]:
dspy.inspect_history()

In [40]:
optimized_program = dspy.load("./sps_dspy_program_gpt-5point2-chat/", allow_pickle=True)

In [41]:
# Example usage: call the optimized program. We must use your get_llm_input to create the prompts.
# The optimized program will still call your configured `lm` under the hood.
i = 63
system_prompt, query_prompt = get_llm_input(data['text_chunk'][i], data['question'][i], data['answer'][i])
out = optimized_program(instruction=system_prompt, prompt=query_prompt)
print("Scores:", out.scores)

In [42]:
data[wanted_cols].iloc[i].to_numpy().astype(int)

In [43]:
test_data = pd.read_csv("sps_dataset_test_split.csv")

In [44]:
pred = []
for i in tqdm(range(test_data.shape[0])):
    system_prompt, query_prompt = get_llm_input(test_data['text_chunk'][i], test_data['question'][i], test_data['answer'][i])
    out = optimized_program(instruction=system_prompt, prompt=query_prompt)
    scores = out.scores.split(" ")
    pred.append(scores)

pred = np.array(pred).astype(int)
pred.shape

In [45]:
(pred == test_data[wanted_cols].to_numpy()).mean()

In [46]:
cohen_kappa_score(test_data[wanted_cols].to_numpy().ravel(), pred.ravel())

In [47]:
k_alpha([test_data[wanted_cols].to_numpy().ravel(), pred.ravel()])

In [48]:
balanced_accuracy_score(test_data[wanted_cols].to_numpy().ravel(), pred.ravel())

In [49]:
print(classification_report(test_data[wanted_cols].to_numpy().ravel(), pred.ravel()))

In [38]:
# optimized_program.save(f"./sps_dspy_program_gpt-5point2-chat/", save_program=True)